# EcoSort Nexus Custom AI Training

Upload your dataset to Google Drive using this folder format:

`MyDrive/EcoSortAI_Project/EcoSortDataset/<Class Name>/*.jpg`

Classes must match the app labels: Paper, Cardboard, Plastic Bottle, Glass Bottle, Metal Can, Food Waste, Banana Peel, Apple, Chip Packet, Mixed Covered Bag.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install tensorflowjs
import os, json, pathlib
import tensorflow as tf

DATA_DIR = '/content/drive/MyDrive/EcoSortAI_Project/EcoSortDataset'
EXPORT_DIR = '/content/drive/MyDrive/EcoSortAI_Project/EcoSortModel_tfjs'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

classes = ['Paper','Cardboard','Plastic Bottle','Glass Bottle','Metal Can','Food Waste','Banana Peel','Apple','Chip Packet','Mixed Covered Bag']
print('Expected classes:', classes)

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    class_names=classes
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    class_names=classes
)

print('Detected class names:', train_ds.class_names)
assert train_ds.class_names == classes, 'Folder names/order must match app labels exactly.'

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.12),
    tf.keras.layers.RandomContrast(0.18),
])

base = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)
base.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x * 255.0)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(len(classes), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3)
]

history = model.fit(train_ds, validation_data=val_ds, epochs=25, callbacks=callbacks)
val_loss, val_acc = model.evaluate(val_ds)
print('Validation accuracy:', round(val_acc * 100, 2), '%')

In [ ]:
# Optional fine tuning if validation accuracy is below target.
base.trainable = True
for layer in base.layers[:-35]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)
val_loss, val_acc = model.evaluate(val_ds)
print('Final validation accuracy:', round(val_acc * 100, 2), '%')

In [ ]:
os.makedirs(EXPORT_DIR, exist_ok=True)
keras_path = '/content/ecosort_model.keras'
model.save(keras_path)
!tensorflowjs_converter --input_format=keras /content/ecosort_model.keras "$EXPORT_DIR"

metadata = {
    'labels': classes,
    'imageSize': 224,
    'validationAccuracy': float(val_acc)
}
with open(os.path.join(EXPORT_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print('Exported TensorFlow.js model to:', EXPORT_DIR)
print('Copy model.json, metadata.json, and *.bin files into app custom-model folder.')